# Lab 06: Deep Residual Networks (ResNet) & Computer Vision Transfer Learning

Welcome to Laboratory 06! In this lab, we explore one of the most influential architectural breakthroughs in deep learning: **Deep Residual Networks (ResNet)** and **Transfer Learning**:
1. **The Degradation Problem**: Why standard deep networks suffer from optimization bottlenecks as depth increases.
2. **Residual Blocks from First Principles**: Implement identity shortcut connections $\mathcal{F}(\mathbf{x}) + \mathbf{x}$ and projection shortcuts.
3. **Pretrained ResNet-18 Transfer Learning**: Fine-tune a convolutional backbone pretrained on 1.2M ImageNet images for new target classification tasks.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import PyTorch, torchvision pretrained models, and data handling tools
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Ensure reproducible initialization
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Residual Block Architecture from Scratch

### Architecture Overview: `ResidualBlock`
The fundamental building block of ResNet solves the vanishing gradient degradation problem by introducing **skip (identity) connections**:
$$\mathbf{y} = \sigma(\mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x})$$

Where $\mathcal{F}(\mathbf{x})$ is the residual mapping:
1. `Conv2d` ($3 \times 3$, stride=$S$) $\to$ `BatchNorm2d` $\to$ `ReLU`
2. `Conv2d` ($3 \times 3$, stride=1) $\to$ `BatchNorm2d`
3. **Shortcut Projection**: If spatial dimensions shrink ($S > 1$) or channel depth changes ($C_{in} \neq C_{out}$), a $1 \times 1$ convolution adjusts $\mathbf{x}$ to match $\mathcal{F}(\mathbf{x})$.
4. **Element-wise Addition**: $\mathcal{F}(\mathbf{x}) + \text{shortcut}(\mathbf{x})$ followed by final `ReLU`.


In [ ]:
# Define the Fundamental ResNet Residual Block Architecture
class ResidualBlock(nn.Module):
    """Residual Block with 2 Convolutions, Batch Normalization, and Identity/Projection Shortcut."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super(ResidualBlock, self).__init__()
        
        # Main Path: First 3x3 Convolution (applies stride for spatial downsampling if stride > 1)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # Main Path: Second 3x3 Convolution (preserves spatial dimensions)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut Branch: Identity shortcut if dimensions match; 1x1 Conv projection if dimensions change
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute shortcut identity or projected representation
        residual = self.shortcut(x)
        
        # Forward pass through main convolutional path
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # Core ResNet Operation: Add residual shortcut to convolutional features
        out += residual
        
        # Final non-linear activation
        out = self.relu(out)
        return out

# Instantiate and verify Residual Block with downsampling (stride=2, 64 channels -> 128 channels)
res_block = ResidualBlock(in_channels=64, out_channels=128, stride=2).to(device)
dummy_input = torch.randn(4, 64, 32, 32).to(device) # Shape: (B=4, C=64, H=32, W=32)
dummy_output = res_block(dummy_input)

print('Input Tensor Shape:        ', dummy_input.shape)
print('Residual Output Tensor Shape:', dummy_output.shape)
assert dummy_output.shape == (4, 128, 16, 16), 'Residual Block output shape mismatch!'
print('[Verification Passed] Residual Block successfully performed 2x spatial downsampling and channel expansion!')


## 3. Transfer Learning with Pretrained ResNet-18 Backbone

### Conceptual Overview: Feature Extraction & Head Replacement
Rather than training a deep network from random scratch on a small dataset:
1. **Load Pretrained Weights**: ImageNet-1k weights contain rich general visual primitives (textures, edges, shapes, semantic parts).
2. **Freeze Feature Extractor**: Set `requires_grad = False` on convolutional backbone layers to prevent destroying pretrained weights.
3. **Replace Classification Head**: Replace the 1,000-class fully-connected head `model.fc` with a new `nn.Linear` layer adapted to our target number of classes.


In [ ]:
# Step 1: Load pre-trained ResNet-18 model with state-of-the-art default ImageNet weights
weights = models.ResNet18_Weights.DEFAULT
resnet18_model = models.resnet18(weights=weights)

# Step 2: Freeze all backbone parameters so gradients are not computed for feature extraction
for param in resnet18_model.parameters():
    param.requires_grad = False

# Step 3: Inspect original classification head and replace with custom 10-class linear classifier
num_in_features = resnet18_model.fc.in_features
print(f'Pretrained ResNet-18 FC Input Feature Dimension: {num_in_features}')

# Replace final linear projection layer (only this layer will be trained!)
resnet18_model.fc = nn.Linear(in_features=num_in_features, out_features=10)
resnet18_model = resnet18_model.to(device)

# Step 4: Verify that only the newly attached classification head parameters are trainable
trainable_params = [name for name, p in resnet18_model.named_parameters() if p.requires_grad]
print('\nTrainable Parameters in Fine-Tuning Setup:')
for p_name in trainable_params:
    print(f'  -> {p_name}')

# Test forward pass with dummy batch
dummy_batch = torch.randn(2, 3, 224, 224).to(device)
output_logits = resnet18_model(dummy_batch)
print(f'\nOutput Logits Shape for 10-class Target: {output_logits.shape}')


## 4. Summary & Practical Transfer Learning Guidelines
1. **Skip Connections**: Ensure gradients propagate directly through the identity path $\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} + \dots$, preventing vanishing gradients in networks with hundreds of layers.
2. **Feature Reuse**: Pretrained backbones drastically reduce training time, data requirements, and computational cost.
3. **Fine-Tuning Strategies**:
   * *Small dataset / high similarity*: Freeze backbone, train linear head only.
   * *Large dataset / low similarity*: Unfreeze deeper layers (e.g. `layer4`) with a small learning rate (e.g. $10^{-4}$).
